# Jets (Modane): are the runs good, and what does the scalar potential change?
The two latest jets arms, 5 seeds each (`jets/launch_jet_SDE.sh`), same set-up except the scalar potential:

| arm | label | terms |
|---|---|---|
| full | `jets_full` | L_6, Scattering 1st/2nd/3rd (Re+Im)/4th Mod2 (Re+Im), **Scalar_morlet_gaussianK** |
| noscalar | `jets_noscalar` | the same without the scalar potential |

Run on Jean Zay from `jets/` (needs `saved_results/`). Sections 1-2 compute here (moments, marginals); section 3 shows the outputs of `python jets/jets_compare.py` (run it first: all statistics of the turbulence comparison, per arm vs the data and arm vs arm).

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
ROOT = Path.cwd()                                   # jets/
sys.path[:0] = [str(ROOT.parent / 'turbulence'), str(ROOT.parent), str(ROOT)]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
import run_comparison as rc
import jets_compare as jc
pd.set_option('display.float_format', '{:.3f}'.format, 'display.width', 250, 'display.max_rows', 100)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})

In [ ]:
ARMS = {'full': 'jets_full', 'noscalar': 'jets_noscalar'}   # {display name: --label}
SEEDS = [500, 501, 502, 503, 504]
OUT = ROOT / 'figures' / 'jets_compare'                   # where jets_compare.py writes
runs = {}                                                  # one entry per (arm, seed)
for arm, label in ARMS.items():
    for s, cfg in jc.find_seed_configs(ROOT, label, SEEDS).items():
        if cfg is None:
            print(f'{arm} seed {s}: no saved run'); continue
        runs[f'{arm} {s}'] = rc.load_config(cfg, ROOT, label=label)
cfg0 = next(iter(runs.values()))['cfg']
x1 = jc.reference_data(cfg0)
xs = {arm: np.concatenate([r['xt'].numpy() for k, r in runs.items() if k.startswith(arm + ' ')]) for arm in ARMS}
J = int(cfg0['J'])
print('data', tuple(x1.shape), '| pooled samples', {k: v.shape for k, v in xs.items()})

## 1. Run health
Run time per seed, and how well each seed matches the statistics it imposes at the end of the run (own-scale error |target - walkers| / max_t |target|; dashed line = 1%).

In [ ]:
display(pd.DataFrame({k: dict(SDE_h=r['runtime_h'], n_samples=len(r['xt'])) for k, r in runs.items()}).T)

In [ ]:
rc.plot_moment_matching(runs)

In [ ]:
rc.moments_above(runs, root=ROOT)          # statistics above 1% at the end, with names

## 2. Wavelet marginals per channel, full vs noscalar (all seeds pooled)
Ratio generated / data of E|W|^2 and of the quantiles of |W| (1 = perfect, dashed), then the tails (density and survival function). Black = data. Real part first, then imaginary part. Only `full` imposes statistics on these marginals directly (Re part, Morlet bank).

In [ ]:
def plot_ratio_lines(bank, part='real', metrics=('m2', 'q0.99', 'q0.999', 'q0.9999'), n_series=10000):
    T = rc.channel_table(x1, xs, bank=bank, J=J, n_series=n_series, part=part)
    cols = rc._colors(xs)
    fig, axes = plt.subplots(1, len(metrics), figsize=(4.3 * len(metrics), 3.6), sharey=True, squeeze=False)
    for ax, m in zip(axes[0], metrics):
        for k in xs:
            ax.plot(np.arange(len(T)), T[m][k].values, marker='o', ms=3.5, lw=1.6, color=cols[k], label=k)
        ax.axhline(1.0, color='k', lw=1, ls='--'); ax.set_title(m, fontsize=10)
        ax.set_xlabel('channel (0 = finest)'); ax.grid(alpha=0.25)
    axes[0, 0].set_ylabel('generated / data'); axes[0, 0].legend(fontsize=8, frameon=False)
    fig.suptitle(f"{bank} bank (Q={rc.BANKS[bank]}), {'Re W' if part == 'real' else 'Im W'}", y=1.02)
    fig.tight_layout(); plt.show()

plot_ratio_lines('morlet')
plot_ratio_lines('psi')

In [ ]:
rc.plot_tail_hists(x1, xs, bank='morlet', J=J, n_ch=J, n_series=10000)

### Imaginary parts

In [ ]:
plot_ratio_lines('morlet', part='imag')
plot_ratio_lines('psi', part='imag')

In [ ]:
rc.plot_tail_hists(x1, xs, bank='morlet', J=J, n_ch=J, n_series=10000, part='imag')

## 3. All statistics: run vs data, and full vs noscalar (from `jets/jets_compare.py`)
**(a) vs data.** Fraction of points within 2σ of the data, per statistic family and arm. σ² = data sampling noise + (seed spread of the arm)² / number of seeds.

**(b) full vs noscalar.** Column 1: fraction of points where the two arms differ by more than 2× the seed noise of their difference. Column 2: among those, fraction where `full` is closer to the data (0.5 = no preference). With 5 seeds per arm, expect ~5-10% of points flagged by chance: read whole families.

In [ ]:
assert (OUT / 'summary.csv').exists(), 'run python jets/jets_compare.py first'
S = pd.read_csv(OUT / 'summary.csv')
display(S.pivot(index='family', columns='run', values='frac_within_2sigma'))
display(Image(OUT / 'summary.png'))

In [ ]:
display(pd.read_csv(OUT / 'arm_difference.csv', comment='#'))
display(Image(OUT / 'arm_difference.png'))

### Cross-scale statistics, bursts, extremes

In [ ]:
for f in ('cross_scale_cuts.png', 'cross_scale_maps.png', 'persistence_extremes.png'):
    display(Image(OUT / f))

### Wavelet bands, Scat4 statistics, single-scale statistics and PDFs

In [ ]:
for f in ('wavelet_scat4.png', 'single_scale.png', 'pdfs.png'):
    display(Image(OUT / f))